# Lab: Draining Tanks and Water Clocks

Math 140H, Honors Calculus I

One physical law guides this whole lab. **Torricelli's law** (1643): water leaves a small hole at the bottom of a tank with speed $\sqrt{2gh}$, where $h$ is the depth of the water above the hole. (This is the speed of a stone dropped from height $h$.) The chain rule turns this law into an equation for the water level. The substitution rule tells us how long draining takes, for any shape of tank. At the end you design an Egyptian water clock.

**How to use this notebook**

- ✏️ **On paper:** work this out by hand first (you will hand these pages in), then run the next cell to check.
- 🔬 **Explore:** change the numbers in the cell, or move the slider, and run the cell again.
- You never need to write code. Only change the inputs you are told to change, then press Shift+Enter.

**Notation.** We measure in cm and s, so $g = 981\ \text{cm/s}^2$.
- $h(t)$ is the **height** of the water at time $t$, and $H = h(0)$ (the starting height).
- $A(h)$ is the **area of the horizontal cross-section of the tank** at height $h$ above the hole.
- $a$ is the **area of the hole**.

As we haven seen in previous examples, the volume of water at depth $h$ is 

$$V(h) = \int_0^h A(s)\,ds.$$


## Torricelli's law: 
*The volume decreases at the rate $a\sqrt{2g\,h(t)}$:*
$$\frac{d}{dt}\,V(h(t)) = -a\sqrt{2g\,h(t)}.$$


#### START BY RUNNING THE NEXT CELL

In [ ]:
#| code-fold: true

#
#
# ALWAYS RUN THIS CELL FIRST!!
#
#

import io
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from IPython.display import display, Math

plt.rcParams['figure.figsize'] = (7, 4.5)
WIDGET_DPI = 72          # on-screen size of the slider pictures: smaller number = smaller picture
G = 981.0                # gravity in cm/s^2
mp.mp.dps = 30

try:
    import ipywidgets as widgets
    _WIDGETS = True
except ImportError:
    _WIDGETS = False


# ---------------------------------------------------------------- sliders
def _slider_view(draw, sliders):
    """Show draw(**slider values) -> (figure, text) and update it in place when a slider moves.

    The figure is rendered to a PNG and swapped into an Image widget of fixed on-screen size,
    so the output area is never cleared and nothing on the page moves."""
    if not _WIDGETS:
        print('(No sliders available here: showing the starting values.)')
        fig, text = draw(**{k: sl.value if hasattr(sl, 'value') else sl for k, sl in sliders.items()})
        plt.show(); print(text)
        return
    img = widgets.Image(format='png')
    txt = widgets.HTML()
    def update(change=None):
        fig, text = draw(**{k: sl.value for k, sl in sliders.items()})
        w_in, h_in = fig.get_size_inches()
        img.layout.width, img.layout.height = f'{round(w_in * WIDGET_DPI)}px', f'{round(h_in * WIDGET_DPI)}px'
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=2 * WIDGET_DPI)   # fixed size: no 'tight' cropping
        plt.close(fig)
        img.value = buf.getvalue()
        txt.value = ('<pre style="font-family: monospace; font-size: 12px; margin: 0">'
                     + text + '</pre>')
    for sl in sliders.values():
        sl.observe(update, names='value')
    update()
    display(widgets.VBox(list(sliders.values()) + [img, txt],
                         layout=widgets.Layout(align_items='flex-start')))


def _slider(description, lo, hi, step, value):
    return widgets.FloatSlider(min=lo, max=hi, step=step, value=value, description=description,
                               readout_format='.2f', continuous_update=True,
                               layout=widgets.Layout(width='400px')) if _WIDGETS else value


# ---------------------------------------------------------------- the simulation
def simulate(A, a, H):
    """Solve h'(t) = -a sqrt(2 g h) / A(h), h(0) = H, numerically (adaptive Runge-Kutta steps).

    Returns (time when the tank is empty, function t -> h(t)). No formula from the lab is used."""
    eps = 1e-12 * H
    def rhs(t, h):
        hh = max(h[0], eps / 2)
        return [-a * np.sqrt(2 * G * hh) / A(hh)]
    def empty(t, h):
        return h[0] - eps
    empty.terminal, empty.direction = True, -1
    sol = solve_ivp(rhs, [0, 1e8], [H * (1 - 1e-12)], events=empty, dense_output=True,
                    rtol=1e-10, atol=1e-14 * H)
    # if the solver stops very close to the bottom of a pointed tank, the time left is negligible
    T = sol.t_events[0][0] if sol.status == 1 else sol.t[-1]
    def h(tt):
        tt = np.clip(np.asarray(tt, dtype=float), 0, T)
        return np.clip(sol.sol(tt)[0], 0, None)
    return T, h


# ---------------------------------------------------------------- Part 1: a bottle
def bottle():
    R, H, r_hole = 4.0, 20.0, 0.2
    A_bottle, a = np.pi * R**2, np.pi * r_hole**2
    T, h = simulate(lambda u: A_bottle, a, H)
    tt = np.linspace(0, T, 800)
    hh = h(tt) / H

    def draw(p):
        y_vals = hh ** p
        line = 1 - tt / T
        fig = plt.figure(figsize=(7, 4.3))
        fig.subplots_adjust(left=0.12, right=0.97, bottom=0.13, top=0.9)
        ax = fig.add_subplot()
        ax.plot(tt, y_vals, lw=2, label=r'$(h(t)/H)^p$')
        ax.plot(tt, line, 'k--', lw=1, label='straight line')
        ax.set_xlim(0, 1.02 * T); ax.set_ylim(0, 1.05)
        ax.set_xlabel('time $t$ (s)'); ax.set_title(f'the bottle: $p = {p:.2f}$'); ax.legend()
        text = (f"simulated time until the bottle is empty:        {T:8.2f} s\n"
                f"largest distance between graph and dashed line:  {np.max(np.abs(y_vals - line)):8.4f}")
        return fig, text

    _slider_view(draw, {'p': _slider('p', 0.1, 2.0, 0.05, 1.0)})


# ---------------------------------------------------------------- Part 2: four tanks
H_RACE, R_HOLE_RACE = 30.0, 0.5
_R0_POT = np.sqrt(4500 * 3 / (13 * 30))        # flower pot: bottom radius r0, top radius 3 r0
TANKS = {
    'cylinder':   lambda u: 150 * np.pi + 0 * u,
    'flower pot': lambda u: np.pi * _R0_POT**2 * (1 + 2 * u / H_RACE)**2,
    'sphere':     lambda u: np.pi * (30 * u - u * u),
    'funnel':     lambda u: np.pi * u * u / 2,
}


def _formula_time(A, a, H):
    """The time from the formula in 2.1 with h1 = 0, by numerical integration."""
    return float(mp.quad(lambda u: A(u) / mp.sqrt(u), [0, H])) / (a * np.sqrt(2 * G))


def race():
    a, H = np.pi * R_HOLE_RACE**2, H_RACE
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6.6), gridspec_kw={'height_ratios': [1, 1.4]})
    off = 0.0
    results = []
    uu = np.linspace(0, H, 400)
    for k, (name, A) in enumerate(TANKS.items()):
        r = np.sqrt(np.clip(A(uu), 0, None) / np.pi)
        w = r.max()
        c = off + w
        ax1.fill_betweenx(uu, c - r, c + r, color=f'C{k}', alpha=0.35)
        ax1.plot(c - r, uu, color=f'C{k}', lw=2); ax1.plot(c + r, uu, color=f'C{k}', lw=2)
        ax1.plot([c - r[0], c + r[0]], [0, 0], color=f'C{k}', lw=2)
        ax1.text(c, -6, name, ha='center', va='top', fontsize=10)
        off += 2 * w + 6
        T, h = simulate(A, a, H)
        tt = np.linspace(0, T, 600)
        ax2.plot(tt, h(tt), lw=2, color=f'C{k}', label=name)
        V = float(mp.quad(A, [0, H]))
        Tf = _formula_time(A, a, H)
        results.append((name, V, T, Tf, Tf * a * np.sqrt(2 * G * H) / V))
    ax1.set_aspect('equal'); ax1.axis('off'); ax1.set_ylim(-12, H + 2)
    ax1.set_title(f'four tanks: height {H:g} cm, hole radius {R_HOLE_RACE * 10:g} mm')
    ax2.set_xlabel('time $t$ (s)'); ax2.set_ylabel('depth $h(t)$ (cm)'); ax2.legend()
    ax2.set_ylim(0, H * 1.03); ax2.set_xlim(0, None)
    plt.tight_layout(); plt.show()
    print(f"{'tank':>11} {'volume (L)':>11} {'simulated T (s)':>16} {'formula 2.1 (s)':>16} {'k':>7}")
    for name, V, T, Tf, k in results:
        print(f"{name:>11} {V / 1000:11.3f} {T:16.2f} {Tf:16.2f} {k:7.4f}")


def time_sums(substituted=False, ns=(10, 100, 1000, 10000, 100000)):
    """Midpoint sums for the emptying time of the flower pot, before or after u = s^2."""
    A, a, H = TANKS['flower pot'], np.pi * R_HOLE_RACE**2, H_RACE
    K = 1 / (a * np.sqrt(2 * G))
    if not substituted:
        f, lo, hi = (lambda u: K * A(u) / np.sqrt(u)), 0.0, H
        display(Math(r"T = \frac{1}{a\sqrt{2g}}\int_0^{H}\frac{A(u)}{\sqrt u}\,du"
                     r"\qquad\text{(flower pot, }H = 30\text{)}"))
    else:
        f, lo, hi = (lambda s: 2 * K * A(s * s)), 0.0, np.sqrt(H)
        display(Math(r"T = \frac{2}{a\sqrt{2g}}\int_0^{\sqrt H} A(s^2)\,ds"
                     r"\qquad\text{(flower pot, }\sqrt H = \sqrt{30}\text{)}"))
    exact = _formula_time(A, a, H)
    print(f"{'n':>8} {'midpoint sum (s)':>18} {'error (s)':>14}")
    for n in ns:
        d = (hi - lo) / n
        mids = lo + d * (np.arange(n) + 0.5)
        S = d * np.sum(f(mids))
        print(f"{n:8d} {S:18.10f} {exact - S:14.3e}")


# ---------------------------------------------------------------- Part 3: water clocks
def clock_shapes():
    """Vessels with radius r(h) = R (h/H)^q; level against time, both scaled to [0, 1]."""
    qs = np.round(np.arange(0, 1.0001, 0.05), 2)
    sims = {}
    for q in qs:
        T, h = simulate(lambda u, q=q: np.pi * u**(2 * q), 1.0, 1.0)
        tt = np.linspace(0, T, 600)
        sims[q] = (tt / T, h(tt))
    hv = np.linspace(0, 1, 300)

    def draw(q):
        q = float(np.round(q / 0.05) * 0.05)
        s, lev = sims[min(qs, key=lambda z: abs(z - q))]
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4), gridspec_kw={'width_ratios': [1, 1.6]})
        fig.subplots_adjust(left=0.05, right=0.97, bottom=0.14, top=0.88, wspace=0.25)
        r = 0.6 * hv ** q
        ax1.fill_betweenx(hv, -r, r, color='C0', alpha=0.3)
        ax1.plot(-r, hv, 'C0', lw=2); ax1.plot(r, hv, 'C0', lw=2)
        ax1.plot([-r[0], r[0]], [0, 0], 'C0', lw=2)
        ax1.set_xlim(-0.7, 0.7); ax1.set_ylim(-0.05, 1.05); ax1.set_aspect('equal'); ax1.axis('off')
        ax1.set_title(f'vessel, $q = {q:.2f}$')
        ax2.plot(s, lev, lw=2, label='water level')
        ax2.plot([0, 1], [1, 0], 'k--', lw=1, label='constant speed')
        ax2.set_xlim(0, 1); ax2.set_ylim(0, 1.03)
        ax2.set_xlabel('time / (time to empty)'); ax2.set_ylabel('depth / H'); ax2.legend()
        text = f"largest distance between the level and the dashed line: {np.max(np.abs(lev - (1 - s))):.4f}"
        return fig, text

    _slider_view(draw, {'q': _slider('q', 0.0, 1.0, 0.05, 0.0)})


def water_clock(R):
    """Simulate a clock of height 25 cm, hole radius 1 mm, radius r(h) = R (h/25)^(1/4)."""
    R, H, a = float(R), 25.0, np.pi * 0.1**2
    T, h = simulate(lambda u: np.pi * R**2 * np.sqrt(u / H), a, H)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.4), gridspec_kw={'width_ratios': [1, 1.5]})
    hv = np.linspace(0, H, 300)
    r = R * (hv / H) ** 0.25
    ax1.fill_betweenx(hv, -r, r, color='C0', alpha=0.25)
    ax1.plot(-r, hv, 'C0', lw=2); ax1.plot(r, hv, 'C0', lw=2)
    for m in range(0, 61, 5):
        if 60 * m <= T * (1 + 1e-6):
            lev = float(h(60 * m))
            rr = R * (lev / H) ** 0.25
            ax1.plot([rr, rr + 0.25 * R], [lev, lev], 'k-', lw=1)
            ax1.text(rr + 0.3 * R, lev, f'{m} min', va='center', fontsize=8)
    ax1.set_xlim(-1.2 * R, 1.9 * R); ax1.set_ylim(-1, H + 1); ax1.set_aspect('equal'); ax1.axis('off')
    ax1.set_title(f'$R = {R:.2f}$ cm: level every 5 min', fontsize=10)
    tt = np.linspace(0, T, 600)
    ax2.plot(tt / 60, h(tt), lw=2, label='water level')
    ax2.plot([0, 60], [H, 0], 'k--', lw=1, label='target: 25 cm in 60 min')
    ax2.set_xlabel('time (min)'); ax2.set_ylabel('depth (cm)'); ax2.legend()
    ax2.set_xlim(0, max(62, 1.05 * T / 60)); ax2.set_ylim(0, H * 1.05)
    plt.tight_layout(); plt.show()
    print(f"time until the clock is empty: {T / 60:.3f} minutes (target: 60)")


def cone_clock():
    """Truncated cones, radius growing by the factor rho from bottom to top: how straight is the level?"""
    rhos = np.round(np.arange(1.0, 6.0001, 0.05), 2)
    sims = {}
    for rho in rhos:
        T, h = simulate(lambda u, rho=rho: np.pi * (1 + (rho - 1) * u)**2, 1.0, 1.0)
        tt = np.linspace(0, T, 1201)
        sims[rho] = (tt / T, h(tt), h(np.arange(13) * T / 12))

    def draw(rho):
        key = min(rhos, key=lambda z: abs(z - rho))
        s, lev, marks = sims[key]
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4), gridspec_kw={'width_ratios': [1, 1.6]})
        fig.subplots_adjust(left=0.03, right=0.97, bottom=0.14, top=0.88, wspace=0.2)
        top = 0.7                                   # drawn top radius; the depth is drawn as 1
        rad = lambda u: top * (1 + (key - 1) * u) / key
        hv = np.linspace(0, 1, 50)
        ax1.fill_betweenx(hv, -rad(hv), rad(hv), color='C0', alpha=0.25)
        ax1.plot(-rad(hv), hv, 'C0', lw=2); ax1.plot(rad(hv), hv, 'C0', lw=2)
        ax1.plot([-rad(0), rad(0)], [0, 0], 'C0', lw=2)
        for m in marks:
            ax1.plot([rad(m), rad(m) + 0.12], [m, m], 'k-', lw=1)
        ax1.set_xlim(-0.8, 0.95); ax1.set_ylim(-0.05, 1.05); ax1.set_aspect('equal'); ax1.axis('off')
        ax1.set_title(f'top/bottom = {key:.2f}: 12 equal time marks', fontsize=10)
        ax2.plot(s, lev, lw=2, label='water level')
        ax2.plot([0, 1], [1, 0], 'k--', lw=1, label='constant speed')
        ax2.set_xlim(0, 1); ax2.set_ylim(0, 1.03)
        ax2.set_xlabel('time / (time to empty)'); ax2.set_ylabel('depth / H'); ax2.legend()
        drops = 100 * (marks[:-1] - marks[1:])
        text = (f"largest distance between the level and the dashed line: {np.max(np.abs(lev - (1 - s))):.4f}\n"
                f"drop of the level in each twelfth of the time (% of H):\n"
                + ' '.join(f'{d:5.1f}' for d in drops))
        return fig, text

    _slider_view(draw, {'rho': _slider('top/bottom', 1.0, 6.0, 0.05, 1.0)})

print('Setup complete.')

## A leaking bottle

✏️ **1.1 On paper.** Explain why $\frac{d}{dt}V(h(t)) = A(h(t))\,h'(t)$. Which two theorems do you use, and what do they need? 

Derive the formula
$$h'(t) = -\frac{a\sqrt{2g\,h(t)}}{A(h(t))}. \qquad (\star)$$

---

Note that this is a **differential equation**, the derivative $h'$ on one side, a formula involving $h$ on the other side.

The first example we look at is a water bottle or soda can, which we assume geometrically is just a cylinder and therefore
has **constant cross-section** $A$. In this case the formula above becomes 

$$h'(t) = -c\sqrt{h(t)}\quad \text{ with } \quad c = \frac aA\sqrt{2g}.$$

---

🔬 **1.2 Explore.** A bottle of radius $4$ cm is filled to $H = 20$ cm and has a hole of radius $2$ mm. 

We can solve the differential equation $(\star)$ numerically, step by step like we did with Euler's method, but with a more accurate update. 
(This method does not use any of the specific formulas from this lab.) 

It plots $\big(h(t)/H\big)^p$ against $t$; Since we are normalizing by diving by $H$, the graph starts at the maximum $1$ at time $t=0$ and reaches $0$ when the bottle is empty. 

For $p = 1$, this is just the scaled water level -- how much water (in percent) is left in the bottle at time $t$. But what happens for other $p$?

Move the slider: for which $p$ is the graph a straight line?

In [ ]:
bottle()

---
✏️ **1.3 On paper.**

1. Use the chain rule and $h' = -c\sqrt h$ to compute $\frac{d}{dt}\sqrt{h(t)}$. Explain how this corresponds to your experiments with the slider in 1.2.
  
2. You can use what you found in (1) to derive an explicit formula for $h(t)$ (no longer a diff. equation): Show that $$h(t) = \big(\sqrt H - \tfrac c2\,t\big)^2$$

3. Check directly, with the chain rule, that this $h(t)$ satisfies $h' = -c\sqrt h$.

4. Show that the bottle is empty at $T = \frac{2\sqrt H}{c}$. Compute $c$ and $T$ for the specific bottle used in 1.2, and compare with the simulated time (obtained by numerically solving the differential equation).

   
---

*You can try this at home:* poke a small hole in a plastic bottle and time it. It takes noticeably longer than $T$ (often by a factor of about $1.6$): the jet narrows just after the hole, so less water gets out than Torricelli's law predicts.

## How long does it take?

For the simple geometric shape of a water bottle, we were able to solve the differential equation, but what about more complicated forms? This is where integration (by substitution) becomes crucial.

---

✏️ **2.1 On paper.** Let $0 < h_1 < H$, and let $T_1$ be the time at which the water height reaches $h_1$. 

By $(\star)$,
$$1 = -\frac{A(h(t))}{a\sqrt{2g\,h(t)}}\;h'(t) \qquad (0\le t\le T_1).$$

Integrate both sides from $0$ to $T_1$, and use the substitution rule to show
$$T_1 = \frac{1}{a\sqrt{2g}}\int_{h_1}^{H}\frac{A(u)}{\sqrt u}\,du.$$

Check the hypotheses of the substitution rule (assume $A$ is continuous). Where do you need $h_1 > 0$?

This gives the draining time *without explicitly solving* $(\star)$.

---

The time until the tank is empty is $$T = \lim_{h_1\to 0} T_1.$$

(If $A(0) > 0$ (a flat bottom), the integrand $\frac{A(u)}{\sqrt u}$ is unbounded near $0$, so this is a limit and not an integral over $[0,H]$. In 2.5 you will show that the limit always exists.) 

---

We can rewrite $T$ as
$$T = k\cdot\frac{V}{a\sqrt{2gH}}, \qquad k = \lim_{h_1\to0}\frac1V\int_{h_1}^H A(u)\,\sqrt{\frac Hu}\,du,$$
where $V$ is the volume of the full tank.

Why would we want to write $T$ this way?

Let's look at the $T_0 = \frac{V}{a\sqrt{2gH}}$: This is how long the tank would take to empty if the water kept leaving at its **initial speed** $\sqrt{2gH}$ (full height $h(0) = H$). The outflow rate would then be the constant $a\sqrt{2gH}$, so the volume $V$ would be gone after $V/(a\sqrt{2gH})$. Because the outflow speed only decreases as the level drops, $T_0$ is a **lower bound for the real draining time**. It contains all the **“physics and size” information** (volume, height, hole size, gravity) and no information about the shape.

$k = T/T_0$, on the other hand is a *pure number* that measures how much the draining slows down, and it depends only on the shape. The cleanest way to see what it is: write it as

$$k = \int_0^H \sqrt{\frac{H}{u}}\;\frac{A(u)\,du}{V}.$$

The factor $\frac{A(u)\,du}{V}$ is the fraction of the water stored in the thin slice at height $u$. While the surface passes through that slice, water leaves at speed $\sqrt{2gu}$ instead of $\sqrt{2gH}$, so draining the slice takes $\sqrt{H/u}$ times longer than at full speed. So $k$ is a **weighted average of the slowdown factor** $\sqrt{H/u}$, weighted by where the water sits.

--- 

✏️ **2.2 On paper.** 

1. Compute $k$ for a cylinder and for a funnel, a cone with its tip at the hole: $A(u) = \pi R^2u^2/H^2$. In each case, compute the integral from $h_1$ to $H$ first, then let $h_1\to0$.

2. Why is $k\ge1$ for every tank? (Hint: compare the integral with $\int_{h_1}^H A(u)\,du = V(H) - V(h_1)$.)

---

✏️ **2.3 Predict, then 🔬 explore.** Four tanks have the same height ($30$ cm), the same volume ($14.1$ L) and the same hole (radius $5$ mm): a cylinder, a flower pot (its radius triples from bottom to top), a sphere, and a funnel. **First write down your prediction:** in which order do they empty?

Then run the cell. The table compares the simulated time with the time from your formula in 2.1.

After running the cell, explain the order with the formula for $k$. Water at which height slows the draining down the most?

In [ ]:
race()

---

🔬 **2.4 Explore.** For the flower pot, the integrand $\frac{A(u)}{\sqrt u}$ in 2.1 is unbounded near $u = 0$. The cell computes midpoint sums for the emptying time $T$.
- By about what factor does the error shrink each time $n$ is multiplied by $10$?
- Why can't we use left sums here?

In [ ]:
time_sums(substituted=False)

---

✏️ **2.5 On paper.** Substitute $u = s^2$ in the formula of 2.1 to show
$$T_1 = \frac{2}{a\sqrt{2g}}\int_{\sqrt{h_1}}^{\sqrt H} A(s^2)\,ds.$$
Why is the new integrand continuous on all of $[0,\sqrt H]$? Use this, and the Fundamental Theorem, to explain why $T_1$ has a limit as $h_1\to0$.

---
🔬 Now run the midpoint sums for the new integral. How fast does the error shrink now?

In [ ]:
time_sums(substituted=True)

## Design a water clock

Egyptian water clocks (from around 1400 BCE) told the time by the water level in a vessel with a hole in the bottom. In a good clock the level falls at constant speed, so that the hour marks are evenly spaced.

🔬 **3.1 Explore.** Take vessels whose radius at height $h$ is $$r(h) = R\,(h/H)^q.$$ For $q = 0$ this is a cylinder, for $q = 1$ a funnel. The cell solves $(\star)$ and plots the level against time, both scaled to $[0,1]$. For which $q$ does the level fall at constant speed?

In [ ]:
clock_shapes()

---

✏️ **3.2 On paper.** Use $(\star)$ to show that the level falls at a constant speed $v$ exactly when
$$A(h) = \frac{a\sqrt{2g}}{v}\,\sqrt h.$$

---

Since $A(h) = r(h)^2 \pi$ and $r(h) = R\,(h/H)^q$, by comparing terms we see that

$$r(h) = R\,(h/H)^{1/4}$$

for a suitable $R$.

--- 

✏️ **3.3 On paper.** Now design a clock: height $H = 25$ cm, hole radius $1$ mm, and the level must fall from $25$ cm to $0$ in exactly one hour. Find $R$.(*Use the formula above. Plug in all the parameters given. What should $v$ be here?*)

---

🔬 **3.4 Explore.** Enter your $R$ (in cm). The cell simulates your clock and marks the water level every 5 minutes. Does your clock keep time?

In [ ]:
R = 10        # <- replace 10 by your R

water_clock(R)

### The Karnak clock

The oldest surviving water clock, from the temple of Karnak, is not shaped like your ideal vessel. It is a truncated cone, wider at the top: roughly $35$ cm deep, with a radius that grows by a factor of about $1.85$ from bottom to top. Its hour marks are drawn on the inside.

🔬 **3.5 Explore.** The slider sets the ratio of top radius to bottom radius of a truncated cone. A ratio of $1$ is a cylinder; the flower pot in 2.3 has ratio $3$.
- For which ratio is the level closest to a straight line? Where does Karnak's ratio $1.85$ sit?
- Look at the 12 marks for equal time steps. For a ratio near $1.85$, which hours are evenly spaced, and which are crowded?

In [ ]:
cone_clock()